In [1]:
# GPT4
!pip install tiktoken
import tiktoken
gpt4Tokenizer = tiktoken.get_encoding('cl100k_base')

# BERT
from transformers import BertTokenizer
bertTokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [3]:
# issue is that they have different tokenizers, so needs to be translated into text and re-tokenized
startingtext = 'Hello, my name is Mike and I like purple.'

# GPT4's tokens:
gpt4Toks = gpt4Tokenizer.encode(startingtext)

# bert's tokens
bertToks = bertTokenizer.encode(startingtext)

print(f'Starting text:\n{startingtext}')
print(f'\n\nGPT4 tokens:\n{gpt4Toks}')
print(f"\nDecoded using GPT4:\n{gpt4Tokenizer.decode(gpt4Toks)}")
print(f"\nDecoded using BERT:\n{bertTokenizer.decode(gpt4Toks)}")

print(f'\n\nBERT tokens:\n{bertToks}')
print(f"\nDecoded using BERT:\n{bertTokenizer.decode(bertToks)}")
print(f"\nDecoded using GPT4:\n{gpt4Tokenizer.decode(bertToks)}")

Starting text:
Hello, my name is Mike and I like purple.


GPT4 tokens:
[9906, 11, 856, 836, 374, 11519, 323, 358, 1093, 25977, 13]

Decoded using GPT4:
Hello, my name is Mike and I like purple.

Decoded using BERT:
lately [unused10] [unused851] [unused831] [unused369] decent [unused318] [unused353] ¾ olympian [unused12]


BERT tokens:
[101, 7592, 1010, 2026, 2171, 2003, 3505, 1998, 1045, 2066, 6379, 1012, 102]

Decoded using BERT:
[CLS] hello, my name is mike and i like purple. [SEP]

Decoded using GPT4:
�.deleteceptionrgretoin\\(idate	for	endinclude�


In [4]:
#  text -> GPT4 tokens -> text -> BERT tokens

# 1) to GPT4 tokens
startingtext = 'Hello, my name is Mike and I like purple.'
gpt4Toks = gpt4Tokenizer.encode(startingtext)

# 2) back to text
gpt4ReconText = gpt4Tokenizer.decode(gpt4Toks)

# 3) then to bert tokens
bertToks = bertTokenizer.encode(gpt4ReconText)

# 4) show the reconstruction
bertTokenizer.decode(bertToks)

'[CLS] hello, my name is mike and i like purple. [SEP]'

In [5]:
# warning about sizes:
txt = 'Lorem ipsum dolor sit amet, consectetur adipiscing elit, sed do eiusmod tempor incididunt ut labore et dolore magna aliqua. Ut enim ad minim veniam, quis nostrud exercitation ullamco laboris nisi ut aliquip ex ea commodo consequat. Duis aute irure dolor in reprehenderit in voluptate velit esse cillum dolore eu fugiat nulla pariatur. Excepteur sint occaecat cupidatat non proident, sunt in culpa qui officia deserunt mollit anim id est laborum.'
print(f'Text contains {len(txt)} characters,')
print(f'              {len(gpt4Tokenizer.encode(txt))} GPT4 tokens, and')
print(f'              {len(bertTokenizer.encode(txt))} Bert tokens.')


Text contains 445 characters,
              96 GPT4 tokens, and
              160 Bert tokens.


In [6]:
# another source of confusion:
txt = 'start\r\n\r\n\r\n\n\r\n\r\n\t\t\t\n\r\n\rend'
# txt = 'start\t\t\t\t\t\t\tend'
# txt = 'start                    end'

bertToks = bertTokenizer.encode(txt)
gpt4Toks = gpt4Tokenizer.encode(txt)

print(f'Reconstruction in BERT:\n  {bertToks}\n  {bertTokenizer.decode(bertToks)}\n')
print(f'Reconstruction in GPT4:\n  {gpt4Toks}\n  {gpt4Tokenizer.decode(gpt4Toks)}')

Reconstruction in BERT:
  [101, 2707, 2203, 102]
  [CLS] start end [SEP]

Reconstruction in GPT4:
  [2527, 881, 81923, 881, 4660, 319, 201, 408]
  start





			

end


In [7]:
bertTokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
gpt4Tokenizer = tiktoken.get_encoding('cl100k_base')

In [9]:
# translation functions
def bert2gpt4(bertToks):
  b = bertTokenizer.decode(bertToks) # decode
  g = gpt4Tokenizer.encode(b) # encode
  return g

def gpt42bert(gpt4Toks):
  g = gpt4Tokenizer.decode(gpt4Toks) # decode
  b = bertTokenizer.encode(g) # encode
  return b[1:-1] # something to exclude special tokens

In [10]:
# just checking that it gives no errors
text = 'I wish chocolate were purple.'

print(bert2gpt4(bertTokenizer.encode(text)))
print(gpt42bert(gpt4Tokenizer.encode(text)))

[58, 88816, 60, 602, 6562, 18414, 1051, 25977, 13, 510, 82476, 60]
[1045, 4299, 7967, 2020, 6379, 1012]


In [11]:
text = "I wanted to paste in a thought-provoking quote here, but I didn't."
print(f'Original text\n  {text}\n')

# initial encoding
bertTox = bertTokenizer.encode(text)
print(f'BERT tokens:\n  {bertTox}\n')

# translate to GPT4
b2g = bert2gpt4(bertTox)
print(f'BERT to GPT4:\n  {gpt4Tokenizer.decode(b2g)}\n')

# back-translate to BERT
back2bert = gpt42bert(b2g)
print(f'Back to BERT:\n  {bertTokenizer.decode(back2bert)}')

Original text
  I wanted to paste in a thought-provoking quote here, but I didn't.

BERT tokens:
  [101, 1045, 2359, 2000, 19351, 1999, 1037, 2245, 1011, 4013, 22776, 14686, 2182, 1010, 2021, 1045, 2134, 1005, 1056, 1012, 102]

BERT to GPT4:
  [CLS] i wanted to paste in a thought - provoking quote here, but i didn ' t. [SEP]

Back to BERT:
  [CLS] i wanted to paste in a thought - provoking quote here, but i didn ' t. [SEP]


In [12]:
# sample text
text = "I still don't have a good quote here. Now it's too late."
print(f'Original text\n  {text}\n')

# initial encoding
gpt4Tox = gpt4Tokenizer.encode(text)
print(f'GPT4 tokens:\n  {gpt4Tox}\n')

# translate to BERT
g2b = gpt42bert(gpt4Tox)
print(f'GPT4 to BERT:\n  {bertTokenizer.decode(g2b)}\n')

# back-translate to GPT4
back2gpt4 = bert2gpt4(g2b)
print(f'Back to GPT4:\n  {gpt4Tokenizer.decode(back2gpt4)}')

Original text
  I still don't have a good quote here. Now it's too late.

GPT4 tokens:
  [40, 2103, 1541, 956, 617, 264, 1695, 12929, 1618, 13, 4800, 433, 596, 2288, 3389, 13]

GPT4 to BERT:
  i still don ' t have a good quote here. now it ' s too late.

Back to GPT4:
  i still don ' t have a good quote here. now it ' s too late.
